In [0]:
# %% Cell 1 — Fetch all isekai anime
import requests, time
import pandas as pd

spark.sql("DROP TABLE IF EXISTS bronze_anilist_isekai")

url = "https://graphql.anilist.co"
query = """
query ($page: Int) {
  Page(page: $page, perPage: 50) {
    pageInfo { hasNextPage }
    media(type: ANIME, tag: "Isekai", sort: POPULARITY_DESC) {
      id
      title { english romaji }
      type
      format
      tags { name }
      status
      startDate { year month day }
      endDate { year month day }
      seasonYear
      season
      averageScore
      popularity
    }
  }
}
"""

rows, page = [], 1
while True:
    r = requests.post(url, json={"query": query, "variables": {"page": page}})
    r.raise_for_status()
    data = r.json()["data"]["Page"]
    rows.extend(data["media"])
    print(f"Page {page}: {len(rows)} total so far")
    if not data["pageInfo"]["hasNextPage"]:
        break
    page += 1
    time.sleep(1)

spark.createDataFrame(rows).write.mode("overwrite").saveAsTable("bronze_anilist_isekai")

print("Saved to bronze_anilist_isekai")

In [0]:
%sql
select distinct format from bronze_anilist_isekai

In [0]:
%sql
select 
id,
title,
format, 
season,
seasonYear,
status,
averageScore,
CASE
    when averageScore >= 85 then 'High'
    when averageScore >= 80 then 'Good'
    when averageScore >= 70 then 'Average'
    when averageScore >= 60 then 'Not Good'
    when averageScore >= 50 then 'Bad'
    else 'Very Bad'
END as grade
from bronze_anilist_isekai
where format in ('TV','ONA','OVA')

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.